# Using Generative Slots



Replace ad-hoc instruct() calls with typed, composable @generative functions.



In this tutorial you learn how to replace manual `instruct()` calls with the `@generative` decorator. This gives you typed function signatures, automatic validation, and composable building blocks that feel like regular Python functions but are backed by LLM generation.



By the end you will have covered:



* Using `@generative` with `Literal` return types for constrained outputs

* Using `@generative` with Pydantic models for structured data

* Composing multiple `@generative` functions into a pipeline

* Injecting shared context with `ChatContext` and `CBlock`

* Adding pre/postcondition checks to generative functions



Prerequisites: Tutorial 01 complete, Mellea installed (`uv add mellea`), and a working backend such as Ollama.



---

## Step 1: From instruct() to @generative with Literal



Start by replacing a basic `instruct()` call with a `@generative` function that returns a constrained `Literal` type.



**Key Concepts:**

- `@generative` decorator converts a function signature into an LLM-backed implementation

- The function's **docstring** becomes the instruction sent to the LLM

- `Literal` return types constrain the output to specific allowed values

- No function body needed - the decorator provides the implementation

- The ellipsis (`...`) indicates the decorator handles everything



This is more maintainable than parsing strings: the type system enforces valid outputs.

In [ ]:
from typing import Literal

import mellea
from mellea import generative


# Define a generative function with Literal return type
# The docstring becomes the LLM instruction
# The return type constrains output to one of three specific values
@generative
def classify_sentiment(text: str) -> Literal["positive", "negative", "mixed"]:
    """Classify the overall sentiment of this customer feedback."""
    # No implementation needed - the @generative decorator handles it
    # The LLM receives: docstring + parameter values
    # It must return exactly one of: "positive", "negative", or "mixed"
    ...


# Initialize Mellea session
m: mellea.MelleaSession = mellea.start_session()

# Call the generative function like any Python function
# The return value is guaranteed to be one of the Literal values
feedback: str = (
    "The onboarding was confusing and took far too long. "
    "Support was helpful once I got through."
)

sentiment: Literal["positive", "negative", "mixed"] = classify_sentiment(m, text=feedback)
print(f"Sentiment: {sentiment}")

# Output will be one of: "positive", "negative", or "mixed"
# The type system ensures no other values are possible

## Step 2: Structured output with Pydantic models



Use Pydantic models as return types to get validated, structured data from the LLM.



**Key Concepts:**

- Pydantic `BaseModel` subclasses define the output schema

- The LLM generates data matching the model's fields and types

- Automatic validation ensures type correctness

- Optional fields use `| None` or `Optional[...]` syntax

- Access fields as attributes: `result.field_name`



This eliminates manual JSON parsing and validation.

In [ ]:
from pydantic import BaseModel

import mellea
from mellea import generative


# Define a Pydantic model for structured output
# The LLM will generate data matching this schema
class FeedbackAnalysis(BaseModel):
    """Structured analysis of customer feedback."""
    
    main_complaint: str  # Required field
    positive_aspect: str | None  # Optional field - may be None
    urgency_level: str  # Required field
    suggested_action: str  # Required field


# Generative function returning a Pydantic model
# The LLM generates structured data, not just text
@generative
def analyze_feedback(feedback: str) -> FeedbackAnalysis:
    """Extract the main complaint, any positive aspects, urgency level, and suggested action from customer feedback."""
    ...


# Initialize session and call the function
m: mellea.MelleaSession = mellea.start_session()

feedback: str = (
    "The onboarding was confusing and took far too long. "
    "Support was helpful once I got through."
)

# The return value is a validated Pydantic model instance
analysis: FeedbackAnalysis = analyze_feedback(m, feedback=feedback)

# Access fields as attributes with full type safety
print(f"Main complaint: {analysis.main_complaint}")
print(f"Positive aspect: {analysis.positive_aspect}")
print(f"Urgency: {analysis.urgency_level}")
print(f"Suggested action: {analysis.suggested_action}")

# Output will vary by model, but structure is guaranteed

## Step 3: Composing multiple generative functions



Build a pipeline by composing multiple `@generative` functions, each with its own typed interface.



**Key Concepts:**

- Each `@generative` function is an independent LLM call

- Functions can call other generative functions

- Data flows through the pipeline with type safety

- No shared state between functions unless explicitly passed

- Easy to test, debug, and modify individual components



This creates modular, reusable building blocks.

In [ ]:
from typing import Literal

from pydantic import BaseModel

import mellea
from mellea import generative


# Define output models
class FeedbackAnalysis(BaseModel):
    """Structured analysis of customer feedback."""
    main_complaint: str
    positive_aspect: str | None
    urgency_level: str


# First generative function: analyze feedback structure
@generative
def analyze_feedback(feedback: str) -> FeedbackAnalysis:
    """Extract the main complaint, any positive aspects, and urgency level from customer feedback."""
    ...


# Second generative function: classify sentiment
@generative
def classify_sentiment(text: str) -> Literal["positive", "negative", "mixed"]:
    """Classify the overall sentiment of this customer feedback."""
    ...


# Third generative function: generate response
@generative
def generate_response(
    complaint: str,
    sentiment: str,
    urgency: str
) -> str:
    """Generate a professional response to the customer addressing their complaint.
    
    Consider the sentiment and urgency level when crafting the response.
    """
    ...


# Orchestration function that composes the pipeline
def process_feedback(m: mellea.MelleaSession, feedback: str) -> dict[str, str]:
    """Complete feedback processing pipeline.
    
    Args:
        m: Active Mellea session
        feedback: Customer feedback text
        
    Returns:
        Dictionary with analysis results and generated response
    """
    # Step 1: Analyze feedback structure (independent LLM call)
    analysis: FeedbackAnalysis = analyze_feedback(m, feedback=feedback)
    
    # Step 2: Classify sentiment (independent LLM call)
    sentiment: Literal["positive", "negative", "mixed"] = classify_sentiment(m, text=feedback)
    
    # Step 3: Generate response using previous results (independent LLM call)
    response: str = generate_response(
        m,
        complaint=analysis.main_complaint,
        sentiment=sentiment,
        urgency=analysis.urgency_level
    )
    
    return {
        "complaint": analysis.main_complaint,
        "positive": analysis.positive_aspect or "None mentioned",
        "urgency": analysis.urgency_level,
        "sentiment": sentiment,
        "response": response
    }


# Run the complete pipeline
m: mellea.MelleaSession = mellea.start_session()

feedback: str = (
    "The onboarding was confusing and took far too long. "
    "Support was helpful once I got through."
)

result: dict[str, str] = process_feedback(m, feedback)

print(f"Complaint: {result['complaint']}")
print(f"Positive: {result['positive']}")
print(f"Urgency: {result['urgency']}")
print(f"Sentiment: {result['sentiment']}")
print(f"Response: {result['response']}")

# Each step is independent and composable

## Step 4: Injecting shared context with ChatContext



Use `ChatContext` and `CBlock` to inject shared instructions or persona across all generative functions in a session.



**Key Concepts:**

- `ChatContext` maintains conversational state across calls

- `CBlock` (Context Block) injects instructions into every generation

- Useful for shared persona, policies, or formatting rules

- All `@generative` functions in the session see the context

- Separates domain logic from cross-cutting concerns



This avoids repeating the same instructions in every function.

In [ ]:
from typing import Literal

import mellea
from mellea import generative
from mellea.stdlib.context import ChatContext
from mellea.core import CBlock

# Define generative functions without persona details
# The persona will be injected via context
@generative
def classify_risk(profile: str) -> Literal["low", "medium", "high"]:
    """Classify the investment risk tolerance based on the client profile."""
    ...


@generative
def recommend_allocation(risk_level: str) -> str:
    """Recommend an asset allocation strategy for the given risk level."""
    ...


# Create a context with shared instructions
# This CBlock will be injected into every generative call
persona_block: CBlock = CBlock(
    """You are a professional financial advisor.
    
    Always:
    - Use clear, jargon-free language
    - Consider the client's age and timeline
    - Emphasize diversification
    - Mention both risks and opportunities
    
    Never:
    - Make specific stock recommendations
    - Guarantee returns
    - Use overly technical terms without explanation
    """
)

# Initialize session with the context
# All generative functions will receive the CBlock instructions
context: ChatContext = ChatContext()
m: mellea.MelleaSession = mellea.start_session(ctx=context)

# Call generative functions - they automatically receive the persona
client_profile: str = (
    "Client is 62, conservative risk tolerance, "
    "needs liquidity within 3 years, concerned about volatility."
)

# Both calls receive the financial advisor persona from context
risk: Literal["low", "medium", "high"] = classify_risk(m, profile=client_profile)
allocation: str = recommend_allocation(m, risk_level=risk)

print(f"Risk Level: {risk}")
print(f"Allocation Strategy: {allocation}")

# The persona is applied consistently without repeating it in each function

In [ ]:
from mellea import generative, start_session
from mellea.stdlib.context import ChatContext
from mellea.core import CBlock

@generative
def grade_essay(essay: str) -> int:
    """Grade the essay and return a score from 1 to 100."""
    ...

@generative
def give_feedback(essay: str) -> list[str]:
    """Return a list of specific improvement suggestions for the essay."""
    ...

essay = "The cat sat on the mat. It was a nice mat. The cat liked it."

m = start_session(ctx=ChatContext())

# No context — grader decides independently.
grade = grade_essay(m, essay=essay)
feedback = give_feedback(m, essay=essay)
print(f"Grade: {grade}")
print(f"Feedback: {feedback}")
# Output will vary — LLM responses depend on model and temperature.

# Inject a persona — both functions now behave as this grader.
m.ctx = m.ctx.add(CBlock(
    "You are an encouraging primary school teacher. "
    "Keep grades above 70 unless there is a serious problem. "
    "Frame all feedback kindly."
))

grade = grade_essay(m, essay=essay)
feedback = give_feedback(m, essay=essay)
print(f"Grade with teacher context: {grade}")
print(f"Feedback with teacher context: {feedback}")
# Output will vary — LLM responses depend on model and temperature.

# Reset and try a different persona.
m.reset()
m.ctx = m.ctx.add(CBlock(
    "You are a grammar specialist. Focus entirely on sentence structure, "
    "punctuation, and vocabulary. Ignore content quality."
))

grade = grade_essay(m, essay=essay)
print(f"Grade with grammar context: {grade}")
# Output will vary — LLM responses depend on model and temperature.

## Step 5: Pre and postcondition checks



Add validation before and after generation using custom checks.



**Key Concepts:**

- Preconditions validate inputs before calling the LLM

- Postconditions validate outputs after generation

- Use `@generative` functions as lightweight validators

- Raise `ValueError` to signal validation failures

- Separates validation logic from generation logic



This adds guardrails without cluttering the main function.

In [1]:
from typing import Literal

import mellea
from mellea import generative


# Precondition check: validate input before generation
@generative
def is_valid_profile(profile: str) -> Literal["valid", "invalid"]:
    """Check if the client profile contains sufficient information for risk assessment.
    
    A valid profile should mention age, risk tolerance, and timeline.
    """
    ...


# Main generative function
@generative
def render_advice(profile: str) -> str:
    """Generate personalized investment advice based on the client profile.
    
    Include specific recommendations for asset allocation and timeline considerations.
    """
    ...


# Postcondition check: validate output after generation
@generative
def mentions_diversification(advice: str) -> Literal["yes", "no"]:
    """Check if the investment advice mentions diversification or spreading risk."""
    ...


# Wrapper function with pre/postcondition checks
def generate_validated_advice(m: mellea.MelleaSession, profile: str) -> str:
    """Generate investment advice with validation checks.
    
    Args:
        m: Active Mellea session
        profile: Client profile information
        
    Returns:
        Validated investment advice
        
    Raises:
        ValueError: If precondition or postcondition fails
    """
    # Precondition: check input validity
    if is_valid_profile(m, profile=profile) == "invalid":
        raise ValueError("Profile lacks sufficient information for risk assessment")
    
    # Generate the advice
    advice: str = render_advice(m, profile=profile)
    
    # Postcondition: check output quality
    if mentions_diversification(m, advice=advice) == "no":
        raise ValueError("Generated advice does not mention diversification")
    
    return advice


# Use the validated function
m: mellea.MelleaSession = mellea.start_session()

profile: str = (
    "Client is 62, conservative risk tolerance, "
    "needs liquidity within 3 years, concerned about volatility."
)

try:
    advice: str = generate_validated_advice(m, profile)
    print(advice)
except ValueError as e:
    print(f"Validation failed: {e}")

# Output will vary by model, but validation ensures quality

=== 17:06:36-INFO ======
Starting Mellea session: backend=ollama, model=granite4:micro, context=SimpleContext
For a 62-year-old client with a conservative risk tolerance who requires liquidity in the next three years and is concerned about volatility, it's recommended to have at least 60% of your portfolio allocated into fixed-income investments such as bonds or CDs (Certificates of Deposit). The remaining 40% can be invested into low-risk, high-yield savings accounts. It's also advised to keep a portion in cash equivalents for easy access and minimize exposure to volatile markets.


## What you built



A pattern for replacing ad-hoc `instruct()` calls with reusable, typed, context-steerable generative functions:



|Pattern|What it gives you|
|---|---|
|`@generative` with `Literal` return|Constrained output, no parsing|
|`@generative` with Pydantic return|Structured output, validated schema|
|Multiple `@generative` functions|Composable pipeline in plain Python|
|`ChatContext` + `CBlock` injection|Shared persona or policy across all functions|
|Pre/postcondition checks|Input validation and output compliance|

---

**See also:** [Generative Functions](https://docs.mellea.ai/guide/generative-functions) | [The Requirements System](https://docs.mellea.ai/concepts/requirements-system) | [Write Custom Verifiers](https://docs.mellea.ai/how-to/write-custom-verifiers)